In [1]:
import urllib.request; exec(urllib.request.urlopen('https://aic-data.aiffel.io/api/colab/setup?t=3vvc47bz').read().decode())

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit



⏳  터널 준비 확인 중...

✅  터널 생성 완료!
🔗  URL: https://attacked-lock-opportunities-promotions.trycloudflare.com

아래 [URL 복사] 버튼을 누른 뒤 웹앱 연결창에 붙여넣으세요. (이 탭은 열어두세요)


✅ 웹앱에 자동 연결 요청을 보냈습니다. 잠시 후 웹앱 화면이 연결됩니다.


In [2]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas scikit-learn matplotlib seaborn -q

import platform
import warnings
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# 재현성: 같은 난수를 항상 같게 만듭니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록)
system = platform.system()
if system == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.0.2
pandas: 2.2.2


# Pandas 벡터화 vs map() vs apply() 완벽 정리

## 📌 내가 이해한 것

### 1. 벡터화 (Vectorization)
우리가 흔히 쓰는 코딩 언어들에 쓰이는 **여러 변수로 자유롭게 연산**

```python
df['return'] = df['close'] - df['open']
df['ratio'] = df['volume'] * df['price']
df['weighted'] = df['high'] * 0.4 + df['low'] * 0.3
```

### 2. map()
- 함수도 쓸 수 있음 (apply와 유사)
- **주요 용도는 맵핑**
- Series 전용

```python
action_map = {1: 'BUY', 0: 'HOLD', -1: 'SELL'}
trades = signal.map(action_map)
```

### 3. apply()
- lambda로 연산식을 바로 쓸 수 있음
- 벡터화가 있으므로 lambda는 거의 의미 없음
- **긴 함수가 필요할 때 사용**

```python
df['level'] = df['value'].apply(lambda x: 'high' if x > 10 else 'low')
```

---

## ❓ 핵심 질문

> 함수를 쓸 수 있는 것이 apply인데, 사실 map도 함수를 쓸 수 있다면 **굳이 apply를 쓸 이유가?**

---

## ✅ 답변: 당신의 지적이 정확합니다!

### Series 입장에서는 apply가 거의 의미 없음

```python
# Series에서는 map이 더 낫다
series.map(함수)    # ✅ 명확, 전문화됨
series.apply(함수)  # 🤔 map과 같은데 뭐하는 건가?
```

---

## 🎯 그럼 apply는 왜 존재하는가?

### DataFrame에서 진가를 발휘합니다!

```python
# Series는 map/apply 모두 같음
s.map(func) == s.apply(func)

# 하지만 DataFrame은?
# ❌ map은 Series 전용 → DataFrame에서 거의 안 씀
df.map()  # 존재하지만 거의 사용 X

# ✅ apply는 DataFrame의 행/열 단위 연산에 필수
df.apply(func, axis=1)  # 행 단위
df.apply(func, axis=0)  # 열 단위
```

---

## 📊 DataFrame에서 apply 필요한 경우

```python
# 여러 컬럼을 동시에 봐야 할 때
def calculate_position(row):
    if row['high'] > row['ma20'] and row['volume'] > row['avg_volume']:
        return row['close'] * 0.02  # 포지션 사이즈
    else:
        return 0

# apply 필수! (map으로는 Series이므로 못 함)
df['position'] = df.apply(calculate_position, axis=1)
```

---

## 📋 최종 정리표

| 상황 | 최선의 선택 | 이유 |
|------|-----------|------|
| **Series 변환** | `map()` | 명확하고 전문화됨 |
| **Series 함수 적용** | `map()` | apply도 되지만 map이 더 낫다 |
| **DataFrame 행/열 연산** | `apply()` | map은 DataFrame에서 동작 안 함 |
| **간단한 연산** | **벡터화** ⭐ | 가장 빠르고 간단 |
| **복잡한 로직** | `apply()` | 긴 함수 정의 후 사용 |

---

## 🏆 결론

1. **Series에서는 map이 더 낫다**
2. **DataFrame에서는 apply가 필수다**
3. **투자 전략 개발에서 가장 중요한 건 벡터화다** ⭐

### 당신의 투자 전략 개발 우선순위

```
1순위: 벡터화 (대부분의 연산)
   ↓
2순위: map() (신호 변환, 매핑)
   ↓
3순위: apply() (여러 컬럼 동시 참조 필요할 때만)
```

In [29]:
# ─────────────────────────────────────────────
# 모두마켓 이번 달 주문 데이터 생성 — 자기완결적 스냅샷
# (지난 노드에서 다룬 오염 요소를 적당히 섞어 둡니다)
# ─────────────────────────────────────────────
np.random.seed(42)
n = 1500

regions = np.random.choice(["서울", "경기", "부산", "인천", "대구"], n, p=[0.4, 0.25, 0.15, 0.1, 0.1])
membership = np.random.choice(["basic", "silver", "gold", "vip"], n, p=[0.5, 0.25, 0.15, 0.1])
channels = np.random.choice(["web", "app", "app ", "APP"], n, p=[0.5, 0.4, 0.05, 0.05])
categories = np.random.choice(["패션", "뷰티", "식품", "가전", "도서"], n)

prices = np.random.choice([9900, 19900, 29900, 49900, 89900, 129900, 249900], n,
                          p=[0.2, 0.25, 0.2, 0.15, 0.1, 0.06, 0.04])
quantities = np.random.choice([1, 1, 1, 2, 2, 3], n)
amount = (prices * quantities).astype(float)

orders = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n + 1)],
    "customer_age": np.random.normal(35, 9, n).round().astype(int),
    "region": regions,
    "membership": membership,
    "channel": channels,
    "category": categories,
    "price": prices.astype(float),
    "quantity": quantities,
    "amount": amount,
})

# 오염 심기: 결측·이상치·표기 혼재
orders.loc[np.random.choice(n, 60, replace=False), "amount"] = np.nan
orders.loc[np.random.choice(n, 30, replace=False), "customer_age"] = np.nan
orders.loc[5, "customer_age"] = 999             # 입력 실수성 이상치
orders.loc[8, "customer_age"] = -3              # 불가능한 음수
orders.loc[12, "quantity"] = 80                 # 비정상적으로 큰 주문
orders.loc[20, "region"] = " 서울 "             # 앞뒤 공백
orders.loc[21, "region"] = "Seoul"              # 영문 표기
orders.loc[40, "membership"] = "VIP"            # 대소문자 혼재

print("이번 달 주문 데이터 준비 완료:", orders.shape)
orders.head()

이번 달 주문 데이터 준비 완료: (1500, 9)


,order_id,customer_age,region,membership,channel,category,price,quantity,amount
0,O00001,30.0,서울,silver,app,패션,19900.0,1,NaN
1,O00002,24.0,대구,basic,app,가전,249900.0,3,749700.0
2,O00003,45.0,부산,basic,web,패션,19900.0,1,19900.0
3,O00004,24.0,경기,basic,app,뷰티,19900.0,1,NaN
4,O00005,41.0,서울,basic,app,패션,19900.0,1,19900.0


In [30]:
# 예제 준비: 표기 정제부터 — 인코딩의 전제
orders["region_clean"] = orders["region"].str.strip().replace({"Seoul": "서울"})
orders["membership_clean"] = orders["membership"].str.lower()
orders["channel_clean"] = orders["channel"].str.strip().str.lower()

print("region 정제 후:", sorted(orders["region_clean"].unique()))
print("membership 정제 후:", sorted(orders["membership_clean"].unique()))
print("channel 정제 후:", sorted(orders["channel_clean"].unique()))

region 정제 후: ['경기', '대구', '부산', '서울', '인천']
membership 정제 후: ['basic', 'gold', 'silver', 'vip']
channel 정제 후: ['app', 'web']


In [31]:
# 스스로 해보자! (2)
# 1) channel_clean One-Hot
ch_oh = pd.get_dummies(orders["channel_clean"], prefix="ch", dtype=int)
orders = pd.concat([orders, ch_oh], axis=1)
print(orders.filter(like="ch_").head())

# 2) drop_first 옵션
cat_oh = pd.get_dummies(orders["category"], prefix="cat", dtype=int)
print(cat_oh.head())
print(cat_oh.shape)
cat_oh = pd.get_dummies(orders["category"], prefix="cat", dtype=int, drop_first=True)
print(cat_oh.head())
print(cat_oh.shape)   # 컬럼이 한 개 줄어든 것을 확인
print('가전이 빠짐: 나머지가 0 이면 가전이므로. 가전이 없어도 가전이라는 것을 알 수 있으므로')

   ch_app  ch_web
0       1       0
1       1       0
2       0       1
3       1       0
4       1       0
   cat_가전  cat_도서  cat_뷰티  cat_식품  cat_패션
0       0       0       0       0       1
1       1       0       0       0       0
2       0       0       0       0       1
3       0       0       1       0       0
4       0       0       0       0       1
(1500, 5)
   cat_도서  cat_뷰티  cat_식품  cat_패션
0       0       0       0       1
1       0       0       0       0
2       0       0       0       1
3       0       1       0       0
4       0       0       0       1
(1500, 4)
가전이 빠짐: 나머지가 0 이면 가전이므로. 가전이 없어도 가전이라는 것을 알 수 있으므로


In [32]:
# ─────────────────────────────────────────────
# 시나리오 0 — 원본 CSV 파일 준비 (실무에서는 외부에서 받아오는 단계)
# ─────────────────────────────────────────────
work_dir = Path("d006_work")
work_dir.mkdir(exist_ok=True)
input_csv  = work_dir / "orders_raw.csv"
output_csv = work_dir / "orders_clean.csv"

orders.to_csv(input_csv, index=False)
print("원본 CSV 저장:", input_csv, "—", input_csv.stat().st_size, "bytes")

원본 CSV 저장: d006_work/orders_raw.csv — 111804 bytes


In [33]:
# 시나리오 1 — 단계 함수들
def load_orders(path):
    # CSV에서 주문 데이터를 읽어 옵니다.
    return pd.read_csv(path)

def clean_strings_full(df):
    # 문자열 컬럼의 공백·대소문자·표기 혼재를 정리합니다.
    return df.assign(
        region=df["region"].str.strip().replace({"Seoul": "서울"}),
        membership=df["membership"].str.lower(),
        channel=df["channel"].str.strip().str.lower(),
    )

def drop_invalid_rows(df, age_min=10, age_max=80, qty_max=10):
    # 불가능한 값과 결측을 제거합니다.
    return (
        df
        .dropna(subset=["amount", "customer_age"])
        .query("@age_min < customer_age < @age_max")
        .query("quantity <= @qty_max")
        .drop_duplicates()
        .reset_index(drop=True)
    )

def add_derived(df):
    # 분석용 파생 컬럼을 추가합니다.
    return df.assign(
        amount_log=lambda d: np.log1p(d["amount"]),
        is_premium=lambda d: d["membership"].isin(["gold", "vip"]).astype(int),
        amount_class=lambda d: np.where(d["amount"] >= 100_000, "high",
                                np.where(d["amount"] >= 30_000, "mid", "low"))
    )

def encode_full(df):
    # membership=Ordinal, 나머지=One-Hot.
    order_map = {"basic": 1, "silver": 2, "gold": 3, "vip": 4}
    out = df.assign(membership_ord=df["membership"].map(order_map))
    one_hots = [
        pd.get_dummies(out["region"],   prefix="region", dtype=int),
        pd.get_dummies(out["channel"],  prefix="ch",     dtype=int),
        pd.get_dummies(out["category"], prefix="cat",    dtype=int),
    ]
    return pd.concat([out] + one_hots, axis=1)

def scale_with_robust(df, cols=("customer_age", "amount", "quantity")):
    # 수치형 컬럼을 Robust로 스케일링하고 _scaled 접미사를 붙입니다.
    scaler = RobustScaler()
    scaled = scaler.fit_transform(df[list(cols)])
    scaled_df = pd.DataFrame(scaled,
                             columns=[f"{c}_scaled" for c in cols],
                             index=df.index)
    return pd.concat([df, scaled_df], axis=1)

print("단계 함수 6개 정의 완료. 다음 셀에서 한 줄로 조립합니다.")

단계 함수 6개 정의 완료. 다음 셀에서 한 줄로 조립합니다.


In [35]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

# 시나리오 2 — end-to-end 파이프라인
def preprocess(input_path):
    # 원본 CSV 경로를 받아 전처리된 DataFrame을 돌려줍니다.
    return (
        load_orders(input_path)
        .pipe(clean_strings_full)
        .pipe(drop_invalid_rows, age_min=10, age_max=80, qty_max=10)
        .pipe(add_derived)
        .pipe(encode_full)
        .pipe(scale_with_robust)
    )

# 진짜 한 줄 호출
clean_df = preprocess(input_csv)
print("입력 행 수 → 출력 행 수:", orders.shape[0], "→", clean_df.shape[0])
print("출력 컬럼 수:", clean_df.shape[1])
clean_df.head(3)

입력 행 수 → 출력 행 수: 1500 → 1404
출력 컬럼 수: 33


,order_id,customer_age,region,membership,channel,category,price,quantity,amount,region_clean,membership_clean,channel_clean,ch_app,ch_web,amount_log,is_premium,amount_class,membership_ord,region_경기,region_대구,region_부산,region_서울,region_인천,ch_app,ch_web,cat_가전,cat_도서,cat_뷰티,cat_식품,cat_패션,customer_age_scaled,amount_scaled,quantity_scaled
0,O00002,24.0,대구,basic,app,가전,249900.0,3,749700.0,대구,basic,app,1,0,13.527430,0,high,1,0,1,0,0,0,1,0,1,0,0,0,0,-0.916667,10.141429,2.0
1,O00003,45.0,부산,basic,web,패션,19900.0,1,19900.0,부산,basic,web,0,1,9.898525,0,low,1,0,0,1,0,0,0,1,0,0,0,0,1,0.833333,-0.284286,0.0
2,O00005,41.0,서울,basic,app,패션,19900.0,1,19900.0,서울,basic,app,1,0,9.898525,0,low,1,0,0,0,1,0,1,0,0,0,0,0,1,0.500000,-0.284286,0.0


In [39]:
import pandas as pd

# ============================================================================
# 품질 리포트 함수
# ============================================================================
def quality_report(before: pd.DataFrame, after: pd.DataFrame) -> dict:
    """
    전처리 전·후 데이터의 품질 차이를 dict로 반환합니다.

    Parameters:
    -----------
    before : pd.DataFrame
        전처리 전 데이터
    after : pd.DataFrame
        전처리 후 데이터

    Returns:
    --------
    dict : 전처리 품질 지표
    """

    report = {
        # 1. 행(Row) 관련 지표
        "rows_before": before.shape[0],
        "rows_after": after.shape[0],
        "removed_pct": round(100 * (1 - after.shape[0] / before.shape[0]), 2),

        # 2. 컬럼(Column) 관련 지표
        "cols_before": before.shape[1],
        "cols_after": after.shape[1],
        "new_cols": after.shape[1] - before.shape[1],

        # 3. 결측치(Missing Values) - 개수
        "missing_count_before": before.isnull().sum().sort_values(ascending=False).head(5).to_dict(),
        "missing_count_after": after.isnull().sum().sort_values(ascending=False).head(5).to_dict(),

        # 4. 결측치(Missing Values) - 비율(%)
        "missing_ratio_before": (
            (before.isnull().sum() / len(before) * 100)
            .sort_values(ascending=False).head(5).round(2).to_dict()
        ),
        "missing_ratio_after": (
            (after.isnull().sum() / len(after) * 100)
            .sort_values(ascending=False).head(5).round(2).to_dict()
        ),

        # 5. 자료형(Data Type) 분포
        "dtypes_after": after.dtypes.value_counts().to_dict(),
    }

    return report


# ============================================================================
# 리포트 출력 함수 (간결한 버전)
# ============================================================================
def print_quality_report(rep: dict):
    """
    데이터 품질 리포트를 간결하게 출력합니다.

    출력 항목:
    1. 입력 행 수 / 출력 행 수 / 제거된 비율
    2. 컬럼별 결측 비율(상위 5개)
    3. 새로 만들어진 파생/인코딩 컬럼 수
    4. 자료형 분포 요약

    Parameters:
    -----------
    rep : dict
        quality_report() 함수 반환값
    """

    print("\n" + "="*70)
    print(" 📊 데이터 품질 리포트")
    print("="*70)

    # 1. 행 처리 요약
    print("\n[1️⃣  행 처리 요약]")
    print(f"  입력: {rep['rows_before']:,}행 → 출력: {rep['rows_after']:,}행 → 제거율: {rep['removed_pct']}%")

    # 2. 컬럼별 결측 비율 (상위 5개)
    print("\n[2️⃣  컬럼별 결측 비율 (상위 5개)]")
    print("  ▶ 전처리 전:")
    if rep['missing_ratio_before']:
        for col, ratio in rep['missing_ratio_before'].items():
            print(f"    - {col}: {ratio}%")
    else:
        print("    - 결측치 없음")

    print("  ▶ 전처리 후:")
    if rep['missing_ratio_after']:
        for col, ratio in rep['missing_ratio_after'].items():
            print(f"    - {col}: {ratio}%")
    else:
        print("    - 결측치 없음")

    # 3. 새로 만들어진 컬럼 수
    print(f"\n[3️⃣  새로 생성된 컬럼 수]")
    print(f"  {rep['new_cols']}개 (파생/인코딩 컬럼)")

    # 4. 자료형 분포
    print(f"\n[4️⃣  자료형 분포 (전처리 후)]")
    if rep['dtypes_after']:
        for dtype, count in rep['dtypes_after'].items():
            print(f"  • {dtype}: {count}개")

    print("\n" + "="*70 + "\n")


# ============================================================================
# 사용 예시
# ============================================================================
if __name__ == "__main__":
    # 테스트 데이터 생성
    import numpy as np

    # 전처리 전 데이터
    orders = pd.DataFrame({
        'customer_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        'customer_age': [25, np.nan, 35, 45, 28, np.nan, 32, 50, 29, 31],
        'amount': [100, 200, 150, np.nan, 300, 250, 180, np.nan, 220, 190],
        'quantity': [1, 2, np.nan, 4, 5, 3, 2, 1, 3, 2],
        'date': pd.date_range('2024-01-01', periods=10)
    })

    # 전처리 후 데이터 (예시)
    clean_df = orders.dropna().copy()
    clean_df['amount_scaled'] = clean_df['amount'] / clean_df['amount'].max()
    clean_df['age_group'] = pd.cut(clean_df['customer_age'], bins=3)

    # 리포트 생성 및 출력
    rep = quality_report(orders, clean_df)
    print_quality_report(rep)

    # 또는 dict로 직접 접근 가능
    print("📌 dict 직접 접근 예시:")
    print(f"  removed_pct: {rep['removed_pct']}%")
    print(f"  new_cols: {rep['new_cols']}")


 📊 데이터 품질 리포트

[1️⃣  행 처리 요약]
  입력: 10행 → 출력: 5행 → 제거율: 50.0%

[2️⃣  컬럼별 결측 비율 (상위 5개)]
  ▶ 전처리 전:
    - customer_age: 20.0%
    - amount: 20.0%
    - quantity: 10.0%
    - customer_id: 0.0%
    - date: 0.0%
  ▶ 전처리 후:
    - customer_id: 0.0%
    - customer_age: 0.0%
    - amount: 0.0%
    - quantity: 0.0%
    - date: 0.0%

[3️⃣  새로 생성된 컬럼 수]
  2개 (파생/인코딩 컬럼)

[4️⃣  자료형 분포 (전처리 후)]
  • float64: 4개
  • int64: 1개
  • datetime64[ns]: 1개
  • category: 1개


📌 dict 직접 접근 예시:
  removed_pct: 50.0%
  new_cols: 2
